Inspired by [Function Calling Sample
](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling?pivots=python)

# Common Libraries and Global Variables

In [1]:
# Common Libraries
import os, sys
from dotenv import load_dotenv # requires python-dotenv
from IPython.display import Markdown, display

config_path = "../../../config" # explicit path to the config folder
sys.path.append(config_path)
from auth import acquire_bearer_token, StaticBearerTokenCredential
if not load_dotenv(f"{config_path}/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

# Global libraries - recall to declare them as "global" in the functions where they are assigned
project_endpoint = "" # must be Foundry V1 project!
deployment_name = ""
bearer_token_cognitiveservices = ""
user_cognitiveservices = ""

Environment variables have been loaded ;-)


# Authentication & Environment setup

In [2]:
def authentication_and_environmentsetup():
    global bearer_token_cognitiveservices, bearer_token_azureai
    bearer_token_cognitiveservices, user_cognitiveservices = acquire_bearer_token(
        scope="https://cognitiveservices.azure.com/.default", # https://ai.azure.com/.default, https://cognitiveservices.azure.com/.default...
        auth_method="default") # default, cli, device

    bearer_token_azureai, _ = acquire_bearer_token(
        scope="https://ai.azure.com/.default", # https://ai.azure.com/.default, https://cognitiveservices.azure.com/.default...
        auth_method="default") # default, cli, device

    print("Bearer token Cognitive Services:", bearer_token_cognitiveservices[:10], "...")
    print("Bearer token Azure AI:", bearer_token_azureai[:10], "...")
    print(f'User info Cognitive Services: {user_cognitiveservices["name"]}, upn: {user_cognitiveservices["upn"]}')
    # bearer_token_cognitiveservices['raw_claims']

authentication_and_environmentsetup()

Bearer token Cognitive Services: eyJ0eXAiOi ...
Bearer token Azure AI: eyJ0eXAiOi ...
User info Cognitive Services: Mauro Minella, upn: mauro.minella@MngEnvMCAP883652.onmicrosoft.com


# Initialization

In [3]:
def init():
    global project_endpoint, deployment_name
    
    project_endpoint = os.getenv("AIF_STD_PROJECT_ENDPOINT") # must be Foundry V1 project
    deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
    
init()

# Create Project Client and OpenAI Client

In [4]:
from azure.ai.projects import AIProjectClient

credential = StaticBearerTokenCredential(bearer_token_azureai)

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
)

openai_client = project_client.get_openai_client()

# Define a function tool for the model to use

In [5]:
def get_horoscope(sign: str) -> str:
    """Generate a horoscope for the given astrological sign."""
    return f"{sign}: Next Tuesday you will befriend a baby otter."


from azure.ai.projects.models import FunctionTool
func_tool = FunctionTool(
    name="get_horoscope",
    parameters={
        "type": "object",
        "properties": {
            "sign": {
                "type": "string",
                "description": "An astrological sign like Taurus or Aquarius",
            },
        },
        "required": ["sign"],
        "additionalProperties": False,
    },
    description="Get today's horoscope for an astrological sign.",
    strict=True,
)

get_horoscope("fish")

'fish: Next Tuesday you will befriend a baby otter.'

# Agent Creation through `PromptAgentDefinition`

In [6]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project_client.agents.create_version(
    agent_name="MyAgent",
    definition=PromptAgentDefinition(
        model=deployment_name,
        instructions="You are a helpful assistant that can use function tools.",
        tools = [func_tool],
    ),
)

In [7]:
# Prompt the model with tools defined
response = openai_client.responses.create(
    input="What is my horoscope? I am an Aquarius.",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

dict(response)

{'id': 'resp_0b93cdab7e0794ab0069ba6bb993388193a1336b6dd6888dcf',
 'created_at': 1773824953.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'You are a helpful assistant that can use function tools.',
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'response',
 'output': [ResponseFunctionToolCall(arguments='{"sign":"Aquarius"}', call_id='call_cwsoyzMm5PR7mTCJ2NKk5lcH', name='get_horoscope', type='function_call', id='fc_0b93cdab7e0794ab0069ba6bba25588193bea98662e34205d5', namespace=None, status='completed', agent_reference={'type': 'agent_reference', 'name': 'MyAgent', 'version': '1'}, response_id='resp_0b93cdab7e0794ab0069ba6bb993388193a1336b6dd6888dcf')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [FunctionTool(name='get_horoscope', parameters={'type': 'object', 'properties': {'sign': {'type': 'string', 'description': 'An astrological sign like Taurus or Aquarius'}}, 'required': ['sign'], 'additionalProperties': False}, strict=

In [8]:
# Process function calls

import json
from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam

input_list: ResponseInputParam = []

for item in response.output: # each item is a ResponseFunctionToolCall object
    if item.type == "function_call":
        print(f"Item type: {type(item)}")
        if item.name == "get_horoscope":
            # Execute the function logic for get_horoscope
            horoscope = get_horoscope(**json.loads(item.arguments))

            # Provide function call results to the model
            input_list.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({"horoscope": horoscope}),
                )
            )
            
input_list

Item type: <class 'openai.types.responses.response_function_tool_call.ResponseFunctionToolCall'>


[{'type': 'function_call_output',
  'call_id': 'call_cwsoyzMm5PR7mTCJ2NKk5lcH',
  'output': '{"horoscope": "Aquarius: Next Tuesday you will befriend a baby otter."}'}]

In [9]:
# Submit function results and get the final response
response = openai_client.responses.create(
    input=input_list,
    previous_response_id=response.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

In [10]:
print(f"Agent response: {response.output_text}")

Agent response: Your horoscope for today as an Aquarius is: "Next Tuesday you will befriend a baby otter." Enjoy your upcoming delightful encounter!


In [11]:
# Clean up resources
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)

{'object': 'agent.version.deleted', 'name': 'MyAgent', 'version': '1', 'deleted': True}